In [5]:
from pathlib import Path

import joblib
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

BASE_DIR = Path.cwd()
DATA_PATH = BASE_DIR / "health_insurance.csv"

if not DATA_PATH.is_file() or DATA_PATH.stat().st_size == 0:
    raise FileNotFoundError("Add a non-empty health_insurance.csv beside this script.")

data = pd.read_csv(DATA_PATH).dropna(subset=["charges"])
if data.empty or "charges" not in data.columns:
    raise ValueError("CSV must contain data and a 'charges' column.")

X = data.drop(columns="charges")
y = pd.to_numeric(data["charges"], errors="raise")

numeric_cols = X.select_dtypes(include="number").columns.tolist()
categorical_cols = X.select_dtypes(exclude="number").columns.tolist()

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore")),
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_cols),
    ("categorical", categorical_pipeline, categorical_cols),
])

model = Pipeline([
    ("preprocessor", preprocessor),
    ("regressor", RandomForestRegressor(
        n_estimators=200, random_state=42, n_jobs=-1
    )),
])
model.fit(X, y)

joblib.dump(model, BASE_DIR / "model.pkl")
joblib.dump(
    model.named_steps["preprocessor"]
         .named_transformers_["numeric"]
         .named_steps["scaler"],
    BASE_DIR / "scaler.pkl",
)

print("Created model.pkl and scaler.pkl")

FileNotFoundError: Add a non-empty health_insurance.csv beside this script.